# Main structural equation models

**REDLINC XII | Computational workflow | R**

Model estimation, group comparisons, fit diagnostics, indirect effects and measurement invariance.

> **Code-only notebook.** This file contains no embedded execution outputs. It requires locally configured inputs and has not been validated end-to-end in a clean environment.


## Before running

1. Install the packages described in the repository README.
2. Configure local input and output locations without committing any local files.
3. Review variable coding, model assumptions and execution order.
4. Keep generated outputs outside version control.


## Analysis block 1

The following code block is preserved from the reviewed draft. Read its internal comments for detailed operations and verify all required inputs before execution.


In [ ]:
# =============================================================================
# SCRIPT SEM V7.0 — FINAL (CORRECTED)
# Changes vs original V7:
#   - OS-aware parallel backend (snow on Windows, multicore on Linux/Mac)
#   - MenoYears diagnostic output
#   - FDR applied separately: BMI family vs covariate family
#   - Table 6 beta calculated via model.frame for precision
#   - Minor output improvements
# =============================================================================

# -----------------------------------------------------------------------------
# 1. INSTALL & LOAD PACKAGES
# -----------------------------------------------------------------------------
#packages <- c("haven", "dplyr", "lavaan", "parallel", "openxlsx")
#new_pkg  <- packages[!packages %in% installed.packages()[, "Package"]]
#if (length(new_pkg) > 0) install.packages(new_pkg, dependencies = TRUE)

library(haven)
library(dplyr)
library(lavaan)
library(parallel)
library(openxlsx)

cat("lavaan version:", as.character(packageVersion("lavaan")), "\n")

# -----------------------------------------------------------------------------
# 2. SETTINGS
# -----------------------------------------------------------------------------
N_BOOT <- 5000

# OS-aware parallel backend
# snow = required on Windows; multicore = more efficient on Linux/Mac (Colab)
parallel_method <- if (.Platform$OS.type == "windows") "snow" else "multicore"
ncores <- max(1, detectCores() - 1)

cat(sprintf("\nMode: FINAL | Bootstrap: %d | Parallel: %s | Cores: %d\n",
            N_BOOT, parallel_method, ncores))
cat("Estimated runtime: 15-30 minutes.\n\n")

out_dir <- Sys.getenv("SEM_WORKDIR", unset = "./")

# -----------------------------------------------------------------------------
# 3. DATA LOADING
# -----------------------------------------------------------------------------
data_file <- paste0(out_dir, "analysis_input.xlsx")
if (!file.exists(data_file)) {
  stop(sprintf("Data file not found at %s.", data_file))
}
dat <- read.xlsx(data_file)
cat(sprintf("Data loaded: %d rows, %d columns.\n", nrow(dat), ncol(dat)))

# Verify required RAW columns exist (derived variables created later are excluded)
required_cols <- c(
  # Jenkins Sleep Scale items
  "Jenkins1","Jenkins2","Jenkins3","Jenkins4",
  # MRS items (MRS3 excluded by design)
  "MRS1","MRS2","MRS4","MRS5","MRS6","MRS7","MRS8","MRS9","MRS10","MRS11",
  # SARC-F items (5 items, each 0/1/2)
  "SARCF","SARCF_A","SARCF_B","SARCF_C","SARCF_D",
  # Outcome
  "MOCATOTAL2",
  # Demographics and covariates
  "Estudio","IMC","Edad","MenoDesde","THMTotal","ActFisica",
  "HipertArt","Diabete","Colesterol",
  # Gynecological surgery components (GynSurgery is derived from these)
  "Histerectom","EdadOoforecto"
  # NOTE: GynSurgery, MRS parcels, SARCF_Total, MenoYears, MOCA_OUT
  # are all derived variables created below — do NOT check them here
)
missing_cols <- required_cols[!required_cols %in% names(dat)]
if (length(missing_cols) > 0) {
  stop(sprintf("Missing columns in xlsx: %s", paste(missing_cols, collapse = ", ")))
}
cat("All required raw columns present.\n\n")

# -----------------------------------------------------------------------------
# 4. DATA PREPARATION
# -----------------------------------------------------------------------------

# Jenkins total (for reference only — latent factor uses items)
dat$JenkinsTotal <- rowSums(dat[, c("Jenkins1","Jenkins2","Jenkins3","Jenkins4")],
                            na.rm = TRUE)

# Gynecological surgery composite
dat$Ooforectomia <- as.integer(!is.na(dat$EdadOoforecto))
dat <- dat %>% mutate(GynSurgery = Histerectom + Ooforectomia)

# MRS parcels (MRS3 excluded — avoids construct overlap with sleep)
dat <- dat %>%
  mutate(
    MRS_Somatic_Limpia = MRS1 + MRS2 + MRS4,
    MRS_Psych_Total    = MRS5 + MRS6 + MRS7 + MRS8,
    MRS_Urogenital_Tot = MRS9 + MRS10 + MRS11
  )

# SARC-F total score (5 items, each 0/1/2; max = 10)
sarcf_items <- c("SARCF","SARCF_A","SARCF_B","SARCF_C","SARCF_D")
dat[sarcf_items] <- lapply(dat[sarcf_items],
                           function(x) as.numeric(as.character(x)))
dat$SARCF_Total <- rowSums(dat[, sarcf_items], na.rm = FALSE)
cat(sprintf("SARCF_Total range: %.0f – %.0f (mean %.2f)\n",
            min(dat$SARCF_Total, na.rm = TRUE),
            max(dat$SARCF_Total, na.rm = TRUE),
            mean(dat$SARCF_Total, na.rm = TRUE)))

# Time since menopause (years)
dat$MenoYears <- dat$Edad - dat$MenoDesde

# ── MenoYears diagnostic ──────────────────────────────────────────────────────
cat(sprintf("\nMenoYears — range: %.1f to %.1f (mean %.1f)\n",
            min(dat$MenoYears, na.rm = TRUE),
            max(dat$MenoYears, na.rm = TRUE),
            mean(dat$MenoYears, na.rm = TRUE)))
n_neg_meno <- sum(dat$MenoYears < 0, na.rm = TRUE)
if (n_neg_meno > 0) {
  cat(sprintf("  ⚠️  %d participants with MenoYears < 0 — check MenoDesde.\n",
              n_neg_meno))
} else {
  cat("  ✅ No negative MenoYears values.\n")
}

# BMI groups (Underweight excluded from main SEM)
dat <- dat %>%
  filter(!is.na(IMC)) %>%
  mutate(
    IMC_Overweight  = as.numeric(IMC >= 25 & IMC < 30),
    IMC_Obese       = as.numeric(IMC >= 30),
    IMC_Group_Label = case_when(
      IMC < 18.5             ~ "Underweight",
      IMC >= 18.5 & IMC < 25 ~ "Normal",
      IMC >= 25 & IMC < 30   ~ "Overweight",
      IMC >= 30              ~ "Obese"
    )
  )

# Exclude underweight from SEM sample
dat_sem <- dat %>% filter(IMC_Group_Label != "Underweight")
cat(sprintf("\nSEM sample (Underweight excluded): N = %d\n", nrow(dat_sem)))
cat("BMI group distribution:\n")
print(table(dat_sem$IMC_Group_Label))

# MoCA education correction (+1 point if ≤12 years of education)
if (!"MoCA_corrected" %in% names(dat_sem)) {
  if (!"MOCATOTAL2" %in% names(dat_sem)) stop("MOCATOTAL2 not found in data.")
  dat_sem$MoCA_corrected <- dat_sem$MOCATOTAL2
  if ("Estudio" %in% names(dat_sem)) {
    dat_sem$MoCA_corrected[!is.na(dat_sem$Estudio) & dat_sem$Estudio <= 12] <-
      dat_sem$MoCA_corrected[!is.na(dat_sem$Estudio) & dat_sem$Estudio <= 12] + 1
    cat(sprintf("\nMoCA education correction applied to %d participants (Estudio ≤ 12).\n",
                sum(!is.na(dat_sem$Estudio) & dat_sem$Estudio <= 12)))
  }
}
dat_sem$MOCA_OUT <- dat_sem$MoCA_corrected

# -----------------------------------------------------------------------------
# 5. BUILD ALIGNED ANALYSIS DATA FRAME
# -----------------------------------------------------------------------------
model_vars <- c("Jenkins1","Jenkins2","Jenkins3","Jenkins4",
                "MRS_Somatic_Limpia","MRS_Psych_Total","MRS_Urogenital_Tot",
                "IMC_Overweight","IMC_Obese",
                "GynSurgery","Edad","MenoYears","THMTotal",
                "HipertArt","Diabete","Colesterol",
                "ActFisica","SARCF_Total","MOCA_OUT","Estudio")

extra_vars <- c("IMC","IMC_Group_Label")
all_vars   <- unique(c(model_vars, extra_vars))

df_analysis <- dat_sem[complete.cases(dat_sem[, all_vars]), all_vars]

# Convert all numeric vars (keep IMC_Group_Label as factor)
numeric_vars <- setdiff(all_vars, "IMC_Group_Label")
df_analysis[numeric_vars] <- lapply(df_analysis[numeric_vars],
                                    function(x) as.numeric(as.character(x)))
df_analysis$IMC_Group_Label <- factor(df_analysis$IMC_Group_Label,
                                      levels = c("Normal","Overweight","Obese"))

cat(sprintf("\nComplete cases (df_analysis): %d\n", nrow(df_analysis)))
cat(sprintf("  Normal:      n = %d\n",
            sum(df_analysis$IMC_Group_Label == "Normal")))
cat(sprintf("  Overweight:  n = %d\n",
            sum(df_analysis$IMC_Group_Label == "Overweight")))
cat(sprintf("  Obese:       n = %d\n",
            sum(df_analysis$IMC_Group_Label == "Obese")))
cat(sprintf("IMC range: %.2f – %.2f (mean %.2f)\n",
            min(df_analysis$IMC), max(df_analysis$IMC),
            mean(df_analysis$IMC)))
cat(sprintf("MenoYears range: %.1f – %.1f (mean %.1f)\n",
            min(df_analysis$MenoYears), max(df_analysis$MenoYears),
            mean(df_analysis$MenoYears)))

# -----------------------------------------------------------------------------
# 6. MODEL SYNTAX
# Indirect effects defined via := so BCa bootstrap CIs are obtained directly
# -----------------------------------------------------------------------------
syntax_v7 <- '
  # A. MEASUREMENT MODEL
  Jenkins_Latent =~ Jenkins1 + Jenkins2 + Jenkins3 + Jenkins4
  MRS_Latent     =~ MRS_Somatic_Limpia + MRS_Psych_Total + MRS_Urogenital_Tot

  # B. STRUCTURAL CASCADE
  # Node 1: Sleep quality
  Jenkins_Latent ~ a_sob*IMC_Overweight + a_obe*IMC_Obese +
                   a_gyn*GynSurgery + a_age*Edad + a_meno*MenoYears +
                   a_thm*THMTotal + a_hyp*HipertArt +
                   a_dia*Diabete + a_dislip*Colesterol

  # Node 2: Menopausal symptom burden
  MRS_Latent     ~ b_sob*IMC_Overweight + b_obe*IMC_Obese +
                   b_slp*Jenkins_Latent + b_gyn*GynSurgery +
                   b_thm*THMTotal + b_act*ActFisica + b_meno*MenoYears +
                   b_hyp*HipertArt + b_dia*Diabete + b_dislip*Colesterol

  # Node 3: Sarcopenia risk
  SARCF_Total    ~ c_sob*IMC_Overweight + c_obe*IMC_Obese +
                   c_mrs*MRS_Latent + c_gyn*GynSurgery +
                   c_age*Edad + c_act*ActFisica + c_meno*MenoYears +
                   c_hyp*HipertArt + c_dia*Diabete + c_dislip*Colesterol

  # Node 4: Cognition (MoCA, education-corrected)
  MOCA_OUT       ~ d_sob*IMC_Overweight + d_obe*IMC_Obese +
                   d_mrs*MRS_Latent + d_srf*SARCF_Total +
                   d_gyn*GynSurgery + d_edu*Estudio + d_age*Edad +
                   d_meno*MenoYears + d_hyp*HipertArt +
                   d_dia*Diabete + d_dislip*Colesterol

  # C. INDIRECT EFFECTS via := (BCa CIs obtained from bootstrap)

  # --- Overweight vs Normal ---
  sob_Sleep_MRS_MoCA       := a_sob * b_slp * d_mrs
  sob_Sleep_MRS_SARCF_MoCA := a_sob * b_slp * c_mrs * d_srf
  sob_MRS_MoCA             := b_sob * d_mrs
  sob_MRS_SARCF_MoCA       := b_sob * c_mrs * d_srf
  sob_SARCF_MoCA           := c_sob * d_srf
  sob_Total_Indirect       := (a_sob*b_slp*d_mrs) + (a_sob*b_slp*c_mrs*d_srf) +
                              (b_sob*d_mrs) + (b_sob*c_mrs*d_srf) + (c_sob*d_srf)
  sob_Total_Effect         := d_sob +
                              (a_sob*b_slp*d_mrs) + (a_sob*b_slp*c_mrs*d_srf) +
                              (b_sob*d_mrs) + (b_sob*c_mrs*d_srf) + (c_sob*d_srf)

  # --- Obese vs Normal ---
  obe_Sleep_MRS_MoCA       := a_obe * b_slp * d_mrs
  obe_Sleep_MRS_SARCF_MoCA := a_obe * b_slp * c_mrs * d_srf
  obe_MRS_MoCA             := b_obe * d_mrs
  obe_MRS_SARCF_MoCA       := b_obe * c_mrs * d_srf
  obe_SARCF_MoCA           := c_obe * d_srf
  obe_Total_Indirect       := (a_obe*b_slp*d_mrs) + (a_obe*b_slp*c_mrs*d_srf) +
                              (b_obe*d_mrs) + (b_obe*c_mrs*d_srf) + (c_obe*d_srf)
  obe_Total_Effect         := d_obe +
                              (a_obe*b_slp*d_mrs) + (a_obe*b_slp*c_mrs*d_srf) +
                              (b_obe*d_mrs) + (b_obe*c_mrs*d_srf) + (c_obe*d_srf)

  # --- Reference pathways (non-BMI) ---
  ref_Surgery_SARCF_MoCA      := c_gyn * d_srf
  ref_Surgery_MRS_MoCA        := b_gyn * d_mrs
  ref_Surgery_MRS_SARCF_MoCA  := b_gyn * c_mrs * d_srf
  ref_THM_Sleep_MRS_MoCA      := a_thm * b_slp * d_mrs
'

# -----------------------------------------------------------------------------
# 7. FIT MODEL WITH BOOTSTRAP
# -----------------------------------------------------------------------------
cat(sprintf("\nFitting SEM (N=%d, bootstrap=%d, parallel=%s, cores=%d)...\n",
            nrow(df_analysis), N_BOOT, parallel_method, ncores))

RNGkind("L'Ecuyer-CMRG")
set.seed(123)

start_time <- proc.time()

fit_boot <- tryCatch({
  if (ncores > 1) {
    sem(model      = syntax_v7,
        data       = df_analysis,
        estimator  = "ML",
        se         = "bootstrap",
        bootstrap  = N_BOOT,
        parallel   = parallel_method,
        ncpus      = ncores,
        fixed.x    = FALSE)
  } else {
    sem(model     = syntax_v7,
        data      = df_analysis,
        estimator = "ML",
        se        = "bootstrap",
        bootstrap = N_BOOT,
        fixed.x   = FALSE)
  }
}, error = function(e) {
  cat("❌ Bootstrap error:", conditionMessage(e), "\n")
  cat("   Falling back to ML (no bootstrap).\n")
  RNGkind("L'Ecuyer-CMRG"); set.seed(123)
  sem(syntax_v7, data = df_analysis, estimator = "ML", fixed.x = FALSE)
})

elapsed <- (proc.time() - start_time)["elapsed"]
cat(sprintf("✅ Done in %.1f minutes.\n", elapsed / 60))

if (!lavInspect(fit_boot, "converged")) {
  warning("⚠️  Model did not converge — check specification.")
} else {
  cat("Model converged.\n")
}

# -----------------------------------------------------------------------------
# 8. EXTRACT RESULTS WITH BCa CIs AND FDR CORRECTION
# -----------------------------------------------------------------------------
all_est <- parameterEstimates(fit_boot,
                              boot.ci.type = "bca.simple",
                              standardized = TRUE)

# ── 8a. Direct regressions ───────────────────────────────────────────────────
reg_all <- all_est %>% filter(op == "~")

# Group N annotation
reg_all$Group_n <- NA
reg_all$Group_n[reg_all$rhs == "IMC_Overweight"] <-
  sum(df_analysis$IMC_Group_Label == "Overweight")
reg_all$Group_n[reg_all$rhs == "IMC_Obese"] <-
  sum(df_analysis$IMC_Group_Label == "Obese")

# FDR SEPARATED BY FAMILY:
# Family 1 — BMI direct effects (primary hypothesis)
bmi_rows <- reg_all$rhs %in% c("IMC_Overweight","IMC_Obese")
reg_all$q_val_bmi    <- NA
reg_all$q_val_global <- NA
reg_all$q_val_bmi[bmi_rows] <- p.adjust(reg_all$pvalue[bmi_rows], method = "BH")
# Family 2 — all direct effects (global, for supplementary reporting)
reg_all$q_val_global <- p.adjust(reg_all$pvalue, method = "BH")

write.csv(reg_all, paste0(out_dir, "SEM_V7_Regresiones.csv"), row.names = FALSE)
cat("\n✅ SEM_V7_Regresiones.csv saved.\n")

# ── 8b. Factor loadings ───────────────────────────────────────────────────────
loadings <- all_est %>% filter(op == "=~")
write.csv(loadings, paste0(out_dir, "SEM_V7_Cargas_Factoriales.csv"),
          row.names = FALSE)
cat("✅ SEM_V7_Cargas_Factoriales.csv saved.\n")

# ── 8c. Indirect effects (via :=; BCa CIs) ───────────────────────────────────
ind_tab <- all_est %>%
  filter(op == ":=") %>%
  select(Effect = lhs, B = est, SE = se, z = z, p = pvalue,
         CI_low = ci.lower, CI_high = ci.upper, Beta = std.all) %>%
  mutate(
    BMI_Group = case_when(
      grepl("^sob_", Effect) ~ "Overweight vs Normal",
      grepl("^obe_", Effect) ~ "Obese vs Normal",
      TRUE                   ~ "Reference (non-BMI)"
    ),
    Pathway     = gsub("^(sob|obe|ref)_", "", Effect),
    q_val       = p.adjust(p, method = "BH"),
    Significant = !(CI_low <= 0 & CI_high >= 0)
  )

ind_tab_out <- ind_tab %>%
  select(BMI_Group, Effect, Pathway, B, SE, p, q_val,
         CI_low, CI_high, Significant)

write.csv(ind_tab_out, paste0(out_dir, "SEM_V7_Efectos_Indirectos.csv"),
          row.names = FALSE)
cat("✅ SEM_V7_Efectos_Indirectos.csv saved.\n")

# ── 8d. Fit indices and R² ────────────────────────────────────────────────────
fi <- fitMeasures(fit_boot, c("chisq","df","pvalue","cfi","tli",
                              "rmsea","rmsea.ci.lower","rmsea.ci.upper","srmr"))
r2_list <- inspect(fit_boot, "rsquare")
vars_r2  <- c("Jenkins_Latent","MRS_Latent","SARCF_Total","MOCA_OUT")
r2_rows  <- data.frame(
  Metric = paste0("R2_", vars_r2),
  Value  = as.numeric(r2_list[vars_r2])
)
fit_tab <- bind_rows(
  data.frame(Metric = names(fi), Value = as.numeric(fi)),
  r2_rows
)
write.csv(fit_tab, paste0(out_dir, "SEM_V7_Indices_Ajuste_R2.csv"),
          row.names = FALSE)
cat("✅ SEM_V7_Indices_Ajuste_R2.csv saved.\n")

cat(sprintf("\nModel fit:\n"))
cat(sprintf("  CFI=%.3f  TLI=%.3f  RMSEA=%.3f [90%% CI: %.3f, %.3f]  SRMR=%.3f\n",
            as.numeric(fi["cfi"]),
            as.numeric(fi["tli"]),
            as.numeric(fi["rmsea"]),
            as.numeric(fi["rmsea.ci.lower"]),
            as.numeric(fi["rmsea.ci.upper"]),
            as.numeric(fi["srmr"])))
cat("R² per equation:\n")
for (i in seq_along(vars_r2)) {
  cat(sprintf("  %-20s R²=%.3f\n", vars_r2[i], r2_rows$Value[i]))
}

# -----------------------------------------------------------------------------
# 9. WALD TESTS (pairwise OW vs OB + joint, with FDR)
# -----------------------------------------------------------------------------
cat("\nRunning Wald tests...\n")

wald_contrasts <- list(
  Sleep_OW_vs_OB   = "a_sob == a_obe",
  MRS_OW_vs_OB     = "b_sob == b_obe",
  SARCF_OW_vs_OB   = "c_sob == c_obe",
  MoCA_OW_vs_OB    = "d_sob == d_obe",
  Sleep_BMI_joint  = "a_sob == 0; a_obe == 0",
  MRS_BMI_joint    = "b_sob == 0; b_obe == 0",
  SARCF_BMI_joint  = "c_sob == 0; c_obe == 0",
  MoCA_BMI_joint   = "d_sob == 0; d_obe == 0"
)

wald_results <- data.frame(
  Test = character(), Type = character(),
  Chi_sq = numeric(), df = integer(), p = numeric(),
  stringsAsFactors = FALSE
)

for (test_name in names(wald_contrasts)) {
  result <- tryCatch({
    wt   <- lavTestWald(fit_boot, constraints = wald_contrasts[[test_name]])
    type <- ifelse(grepl("joint", test_name),
                   "Joint (All BMI=0)", "Pairwise (OW vs OB)")
    data.frame(Test = test_name, Type = type,
               Chi_sq = round(wt$stat, 4),
               df = wt$df,
               p  = round(wt$p.value, 4),
               stringsAsFactors = FALSE)
  }, error = function(e) {
    data.frame(Test = test_name, Type = "ERROR",
               Chi_sq = NA, df = NA, p = NA,
               stringsAsFactors = FALSE)
  })
  wald_results <- rbind(wald_results, result)
}

# FDR for valid Wald p-values
valid_p <- !is.na(wald_results$p)
wald_results$q_val <- NA
wald_results$q_val[valid_p] <- p.adjust(wald_results$p[valid_p], method = "BH")
wald_results$Sig <- case_when(
  is.na(wald_results$p)                                      ~ "ERROR",
  wald_results$p < 0.05 & wald_results$q_val < 0.10         ~ "* p<.05 (FDR q<.10)",
  wald_results$p < 0.05 & wald_results$q_val >= 0.10        ~ "* p<.05 (FDR ns)",
  wald_results$p >= 0.05                                     ~ "ns"
)

cat("\nWald test results:\n")
print(wald_results, row.names = FALSE)
write.csv(wald_results, paste0(out_dir, "SEM_V7_Wald_Tests.csv"),
          row.names = FALSE)
cat("✅ SEM_V7_Wald_Tests.csv saved.\n")

# -----------------------------------------------------------------------------
# 10. TABLE 5 — Sensitivity: continuous BMI (linear + quadratic)
# -----------------------------------------------------------------------------
cat("\nRunning sensitivity analysis (continuous BMI)...\n")

df_analysis$IMC_c  <- as.numeric(scale(df_analysis$IMC,
                                        center = TRUE, scale = FALSE))
df_analysis$IMC_c2 <- df_analysis$IMC_c^2

syntax_sens <- '
  Jenkins_Latent =~ Jenkins1 + Jenkins2 + Jenkins3 + Jenkins4
  MRS_Latent     =~ MRS_Somatic_Limpia + MRS_Psych_Total + MRS_Urogenital_Tot

  Jenkins_Latent ~ IMC_c + IMC_c2 + GynSurgery + Edad + MenoYears +
                   THMTotal + HipertArt + Diabete + Colesterol

  MRS_Latent     ~ IMC_c + IMC_c2 + Jenkins_Latent + GynSurgery + THMTotal +
                   ActFisica + MenoYears + HipertArt + Diabete + Colesterol

  SARCF_Total    ~ IMC_c + IMC_c2 + MRS_Latent + GynSurgery + Edad +
                   ActFisica + MenoYears + HipertArt + Diabete + Colesterol

  MOCA_OUT       ~ IMC_c + IMC_c2 + MRS_Latent + SARCF_Total + GynSurgery +
                   Estudio + Edad + MenoYears + HipertArt + Diabete + Colesterol
'

fit_sens <- tryCatch(
  sem(syntax_sens, data = df_analysis, estimator = "MLR", fixed.x = TRUE),
  error = function(e) {
    cat("⚠️  Sensitivity error:", conditionMessage(e), "\n"); NULL
  }
)

if (!is.null(fit_sens)) {
  sens_est <- parameterEstimates(fit_sens, standardized = TRUE) %>%
    filter(op == "~", rhs %in% c("IMC_c","IMC_c2")) %>%
    select(Dependent = lhs, Term = rhs, B = est, SE = se,
           p = pvalue, Beta = std.all)
  sens_est$q_val <- p.adjust(sens_est$p, method = "BH")
  write.csv(sens_est, paste0(out_dir, "SEM_V7_Sensibilidade_Continuo.csv"),
            row.names = FALSE)
  cat("✅ SEM_V7_Sensibilidade_Continuo.csv saved.\n")

  # Report significant quadratic terms
  quad_sig <- sens_est %>% filter(Term == "IMC_c2", p < 0.05)
  if (nrow(quad_sig) > 0) {
    cat("  Significant quadratic (IMC²) terms:\n")
    for (i in 1:nrow(quad_sig)) {
      cat(sprintf("    %s: B=%.4f, β=%.4f, p=%.4f\n",
                  quad_sig$Dependent[i], quad_sig$B[i],
                  quad_sig$Beta[i], quad_sig$p[i]))
    }
  } else {
    cat("  No significant quadratic terms — linear cascade assumption supported.\n")
  }
}

# -----------------------------------------------------------------------------
# 11. TABLE 6 — Total association of BMI with cognition (no mediators)
# -----------------------------------------------------------------------------
cat("\nRunning total association models (Table 6)...\n")

covariates_str <- paste(
  "Edad + Estudio + MenoYears + THMTotal + GynSurgery + ActFisica +",
  "HipertArt + Diabete + Colesterol"
)

# Helper: standardized beta from model.frame (avoids sample mismatch)
std_beta <- function(fit, term) {
  mf <- model.frame(fit)
  coef(fit)[term] * sd(mf[[term]]) / sd(mf[[all.vars(formula(fit))[1]]])
}

# Model 1: linear BMI
fit_lm1 <- lm(as.formula(paste("MOCA_OUT ~ IMC_c +", covariates_str)),
               data = df_analysis)
s1  <- summary(fit_lm1)
ci1 <- confint(fit_lm1)

# Model 2: linear + quadratic BMI
fit_lm2 <- lm(as.formula(paste("MOCA_OUT ~ IMC_c + IMC_c2 +", covariates_str)),
               data = df_analysis)
s2  <- summary(fit_lm2)
ci2 <- confint(fit_lm2)

# Model 3: categorical BMI (OW + OB dummies)
fit_lm3 <- lm(as.formula(paste("MOCA_OUT ~ IMC_Overweight + IMC_Obese +",
                                covariates_str)),
               data = df_analysis)
s3  <- summary(fit_lm3)
ci3 <- confint(fit_lm3)

total_assoc <- data.frame(
  Model = c("Linear BMI",
            "Quadratic BMI","Quadratic BMI",
            "Categorical BMI","Categorical BMI"),
  Term  = c("IMC_c (linear)",
            "IMC_c (linear)","IMC_c2 (quadratic)",
            "IMC_Overweight","IMC_Obese"),
  B     = c(coef(fit_lm1)["IMC_c"],
            coef(fit_lm2)["IMC_c"],  coef(fit_lm2)["IMC_c2"],
            coef(fit_lm3)["IMC_Overweight"], coef(fit_lm3)["IMC_Obese"]),
  SE    = c(s1$coefficients["IMC_c","Std. Error"],
            s2$coefficients["IMC_c","Std. Error"],
            s2$coefficients["IMC_c2","Std. Error"],
            s3$coefficients["IMC_Overweight","Std. Error"],
            s3$coefficients["IMC_Obese","Std. Error"]),
  p     = c(s1$coefficients["IMC_c","Pr(>|t|)"],
            s2$coefficients["IMC_c","Pr(>|t|)"],
            s2$coefficients["IMC_c2","Pr(>|t|)"],
            s3$coefficients["IMC_Overweight","Pr(>|t|)"],
            s3$coefficients["IMC_Obese","Pr(>|t|)"]),
  Beta  = c(std_beta(fit_lm1, "IMC_c"),
            std_beta(fit_lm2, "IMC_c"),  std_beta(fit_lm2, "IMC_c2"),
            NA, NA),
  CI_low  = c(ci1["IMC_c",1],
              ci2["IMC_c",1],  ci2["IMC_c2",1],
              ci3["IMC_Overweight",1], ci3["IMC_Obese",1]),
  CI_high = c(ci1["IMC_c",2],
              ci2["IMC_c",2],  ci2["IMC_c2",2],
              ci3["IMC_Overweight",2], ci3["IMC_Obese",2]),
  R2_model = c(s1$r.squared, s2$r.squared, s2$r.squared,
               s3$r.squared, s3$r.squared),
  stringsAsFactors = FALSE
)
total_assoc$q_val <- p.adjust(total_assoc$p, method = "BH")

write.csv(total_assoc, paste0(out_dir, "SEM_V7_Asociacion_Total_MoCA.csv"),
          row.names = FALSE)
cat("✅ SEM_V7_Asociacion_Total_MoCA.csv saved.\n")

# -----------------------------------------------------------------------------
# 12. FINAL SUMMARY
# -----------------------------------------------------------------------------
cat("\n", paste(rep("=", 70), collapse = ""), "\n")
cat("ANALYSIS COMPLETED — SEM V7.0 (CORRECTED)\n")
cat(paste(rep("=", 70), collapse = ""), "\n")
cat(sprintf("SEM sample: N=%d | Bootstrap: %d | Seed: 123\n",
            nrow(df_analysis), N_BOOT))
cat(sprintf("Parallel: %s (%d cores)\n", parallel_method, ncores))
cat(sprintf("\nFit: CFI=%.3f TLI=%.3f RMSEA=%.3f [%.3f, %.3f] SRMR=%.3f\n",
            as.numeric(fi["cfi"]), as.numeric(fi["tli"]),
            as.numeric(fi["rmsea"]),
            as.numeric(fi["rmsea.ci.lower"]),
            as.numeric(fi["rmsea.ci.upper"]),
            as.numeric(fi["srmr"])))

cat("\nSignificant indirect effects (CI excludes 0):\n")
sig_ind <- ind_tab_out %>% filter(Significant == TRUE)
if (nrow(sig_ind) > 0) {
  for (i in 1:nrow(sig_ind)) {
    cat(sprintf("  %-22s | %-30s B=%.4f [%.3f, %.3f] p=%.4f q=%.4f\n",
                sig_ind$BMI_Group[i], sig_ind$Pathway[i],
                sig_ind$B[i], sig_ind$CI_low[i],
                sig_ind$CI_high[i], sig_ind$p[i], sig_ind$q_val[i]))
  }
} else {
  cat("  None significant at 95% CI.\n")
}

cat("\nOutput files saved to", out_dir, ":\n")
files <- list.files(out_dir, pattern = "^SEM_V7_.*\\.csv$")
for (f in files) cat(sprintf("  %s\n", f))
cat(paste(rep("=", 70), collapse = ""), "\n")

## Analysis block 2

The following code block is preserved from the reviewed draft. Read its internal comments for detailed operations and verify all required inputs before execution.


In [ ]:
# =============================================================================
# CELL 2 — MEASUREMENT INVARIANCE (Normal, Overweight, Obese) — V7 CORRECTED
# Changes vs original:
#   - estimator = "MLR" for robust fit indices
#   - Invariance_Supported column added (ΔCFI < .010 & ΔRMSEA < .015)
#   - lavTestLRT added for likelihood ratio tests between nested models
#   - Clear direction annotation for ΔCFI (negative = deterioration)
#
# REQUIRES: df_analysis from Cell 1 (includes IMC_Group_Label)
# NOTE: Underweight already excluded from df_analysis by Cell 1 filter.
#       Invariance tests 3 groups: Normal, Overweight, Obese.
# =============================================================================

if (!require(semTools)) install.packages("semTools", dependencies = TRUE)
library(semTools)
library(lavaan)

# ── Pre-flight checks ─────────────────────────────────────────────────────────
if (!exists("df_analysis")) {
  stop("df_analysis not found. Run Cell 1 first.")
}
if (!"IMC_Group_Label" %in% names(df_analysis)) {
  stop("IMC_Group_Label missing from df_analysis. Check Cell 1.")
}
if (!exists("out_dir")) out_dir <- Sys.getenv("SEM_WORKDIR", unset = "./")

df_inv <- df_analysis
df_inv$IMC_Group_Label <- factor(df_inv$IMC_Group_Label,
                                  levels = c("Normal","Overweight","Obese"))

cat("Groups for invariance analysis:", levels(df_inv$IMC_Group_Label), "\n")
cat("Sample sizes per group:\n")
print(table(df_inv$IMC_Group_Label))

# ── Measurement model ─────────────────────────────────────────────────────────
model_measurement <- '
  Jenkins_Latent =~ Jenkins1 + Jenkins2 + Jenkins3 + Jenkins4
  MRS_Latent     =~ MRS_Somatic_Limpia + MRS_Psych_Total + MRS_Urogenital_Tot
'

# ── Fit invariance sequence ───────────────────────────────────────────────────
# estimator = "MLR" for robust fit indices (Satorra-Bentler scaled chi-square)
# consistent with the main SEM estimator

cat("\nFitting configural model (MLR)...\n")
fit_config <- cfa(model_measurement,
                  data      = df_inv,
                  group     = "IMC_Group_Label",
                  estimator = "MLR")

cat("Fitting metric model (loadings constrained equal across groups)...\n")
fit_metric <- cfa(model_measurement,
                  data        = df_inv,
                  group       = "IMC_Group_Label",
                  group.equal = "loadings",
                  estimator   = "MLR")

cat("Fitting scalar model (loadings + intercepts constrained equal)...\n")
fit_scalar <- cfa(model_measurement,
                  data        = df_inv,
                  group       = "IMC_Group_Label",
                  group.equal = c("loadings","intercepts"),
                  estimator   = "MLR")

# ── Fit index extraction ──────────────────────────────────────────────────────
get_fit <- function(fit, model_name) {
  # Use robust indices for MLR estimator
  m <- fitMeasures(fit, c("chisq.scaled","df.scaled","pvalue.scaled",
                           "cfi.robust","tli.robust",
                           "rmsea.robust","srmr"))
  data.frame(
    Model  = model_name,
    chisq  = round(m["chisq.scaled"], 3),
    df     = round(m["df.scaled"],    0),
    p      = round(m["pvalue.scaled"],4),
    CFI    = round(m["cfi.robust"],   3),
    TLI    = round(m["tli.robust"],   3),
    RMSEA  = round(m["rmsea.robust"], 3),
    SRMR   = round(m["srmr"],         3),
    stringsAsFactors = FALSE
  )
}

fit_summary <- rbind(
  get_fit(fit_config, "1_Configural"),
  get_fit(fit_metric, "2_Metric"),
  get_fit(fit_scalar, "3_Scalar")
)

cat("\nMeasurement invariance fit indices (MLR robust):\n")
print(fit_summary, row.names = FALSE)

# ── Delta CFI and RMSEA ───────────────────────────────────────────────────────
# Negative ΔCFI = deterioration when adding constraints (expected)
# Criterion: |ΔCFI| < .010 AND |ΔRMSEA| < .015 (Chen, 2007)

diff_df <- data.frame(
  Comparison = c("Configural → Metric", "Metric → Scalar"),
  dCFI   = c(
    fit_summary$CFI[2]   - fit_summary$CFI[1],   # negative if worse
    fit_summary$CFI[3]   - fit_summary$CFI[2]
  ),
  dRMSEA = c(
    fit_summary$RMSEA[2] - fit_summary$RMSEA[1], # positive if worse
    fit_summary$RMSEA[3] - fit_summary$RMSEA[2]
  )
)

# Invariance supported if |ΔCFI| < .010 AND |ΔRMSEA| < .015
diff_df$Invariance_Supported <- abs(diff_df$dCFI)   < 0.010 &
                                 abs(diff_df$dRMSEA) < 0.015

diff_df$dCFI   <- round(diff_df$dCFI,   3)
diff_df$dRMSEA <- round(diff_df$dRMSEA, 3)

cat("\nInvariance differences (Chen, 2007 criteria: |ΔCFI| < .010, |ΔRMSEA| < .015):\n")
print(diff_df, row.names = FALSE)

# ── Likelihood ratio tests ────────────────────────────────────────────────────
# For MLR, lavTestLRT uses the Satorra-Bentler scaled difference test
cat("\nLikelihood ratio tests (Satorra-Bentler scaled):\n")

lrt_metric <- tryCatch(
  lavTestLRT(fit_config, fit_metric),
  error = function(e) { cat("LRT Config vs Metric error:", conditionMessage(e), "\n"); NULL }
)

lrt_scalar <- tryCatch(
  lavTestLRT(fit_metric, fit_scalar),
  error = function(e) { cat("LRT Metric vs Scalar error:", conditionMessage(e), "\n"); NULL }
)

if (!is.null(lrt_metric)) {
  cat("\nConfigural vs Metric:\n")
  print(lrt_metric)
}
if (!is.null(lrt_scalar)) {
  cat("\nMetric vs Scalar:\n")
  print(lrt_scalar)
}

# ── Determine group.equal constraints for main model ─────────────────────────
# The level of invariance supported determines which constraints to apply
# in the primary SEM when run as multigroup (if needed).
metric_ok <- diff_df$Invariance_Supported[1]
scalar_ok <- diff_df$Invariance_Supported[2]

cat("\nInvariance decision:\n")
if (scalar_ok) {
  cat("  ✅ Scalar invariance supported — loadings + intercepts can be constrained.\n")
  group_constraints <- c("loadings","intercepts")
} else if (metric_ok) {
  cat("  ⚠️  Metric invariance only — loadings can be constrained; intercepts free.\n")
  group_constraints <- "loadings"
} else {
  cat("  ❌ Configural only — no constraints supported; interpret comparisons cautiously.\n")
  group_constraints <- character(0)
}
cat(sprintf("  → Recommended group.equal: %s\n",
            if (length(group_constraints) == 0) "none"
            else paste(group_constraints, collapse = " + ")))

# ── Export results ────────────────────────────────────────────────────────────
write.csv(fit_summary,
          paste0(out_dir, "SEM_V7_Invariance_FitSummary.csv"),
          row.names = FALSE)
write.csv(diff_df,
          paste0(out_dir, "SEM_V7_Invariance_Differences.csv"),
          row.names = FALSE)

# Export LRT results if available
if (!is.null(lrt_metric) && !is.null(lrt_scalar)) {
  lrt_out <- data.frame(
    Comparison = c("Configural vs Metric","Metric vs Scalar"),
    Chi_sq_diff = c(
      as.numeric(lrt_metric[2, "Chisq diff"]),
      as.numeric(lrt_scalar[2, "Chisq diff"])
    ),
    df_diff = c(
      as.numeric(lrt_metric[2, "Df diff"]),
      as.numeric(lrt_scalar[2, "Df diff"])
    ),
    p = c(
      as.numeric(lrt_metric[2, "Pr(>Chisq)"]),
      as.numeric(lrt_scalar[2, "Pr(>Chisq)"])
    )
  )
  write.csv(lrt_out,
            paste0(out_dir, "SEM_V7_Invariance_LRT.csv"),
            row.names = FALSE)
  cat("\n✅ LRT results saved.\n")
}

cat("\n✅ Invariance results saved to", out_dir, ":\n")
cat("  SEM_V7_Invariance_FitSummary.csv\n")
cat("  SEM_V7_Invariance_Differences.csv\n")
cat("  SEM_V7_Invariance_LRT.csv\n")

cat("\n", paste(rep("=", 65), collapse=""), "\n")
cat("INVARIANCE ANALYSIS COMPLETE — V7 (MLR, 3 groups)\n")
cat(paste(rep("=", 65), collapse=""), "\n")

## Reproducibility and interpretation

Check execution logs, convergence and assumptions before interpreting any estimates. This repository intentionally excludes study-specific outputs. For scientific interpretation, refer to the associated publication when available.
